# dataguard-deid — Quickstart

This notebook walks through all major features of the `dataguard_deid` library:

1. Installation & import
2. Detect PII in plain text (`analyze.text`)
3. Anonymize plain text — all three modes (`guard.text`)
4. Work with documents — `.txt`, `.pdf`, `.docx` (`analyze.doc` / `guard.doc`)
5. Custom patterns
6. Config options — entity filters, score threshold

## 1. Installation

```bash
pip install dataguard-deid
```

Or from the repo root:

```bash
pip install -e .
```

In [ ]:
import dataguard_deid
from dataguard_deid import analyze, guard, custom_pattern, ALL_NL_ENTITY_TYPES

print("dataguard_deid version:", dataguard_deid.__version__)

## 2. Detect PII in plain text

`analyze.text()` returns a list of findings. Each finding contains:

| Key | Description |
|-----|-------------|
| `type` | Entity label (e.g. `BSN`, `EMAIL_ADDRESS`) |
| `start` | Start character offset |
| `end` | End character offset |
| `score` | Confidence score (0.0 – 1.0) |

In [ ]:
sample_text = (
    "Patiënt: Jan de Vries, BSN 999999990. "
    "IBAN: NL91 ABNA 0417 1643 00. "
    "E-mail: jan.devries@umcg.nl. "
    "Telefoon: 06-12345678."
)

findings = analyze.text(sample_text)

for f in findings:
    print(f"{f['type']:<20} {sample_text[f['start']:f['end']]:<30} score={f['score']:.2f}")

### All supported entity types

In [ ]:
print(f"{len(ALL_NL_ENTITY_TYPES)} supported entity types:")
print(", ".join(sorted(ALL_NL_ENTITY_TYPES)))

## 3. Anonymize plain text — three modes

`guard.text()` returns a dict with two keys:
- `guarded_text` — the text with PII replaced
- `findings` — list of detected spans (same structure as `analyze.text`, plus `original_text`)

### Mode: `anonymize` (default) — realistic synthetic replacements

In [ ]:
result = guard.text(sample_text, config={"mode": "anonymize"})

print("Original:")
print(sample_text)
print()
print("Guarded:")
print(result["guarded_text"])

### Mode: `tag` — bracket labels

In [ ]:
result_tag = guard.text(sample_text, config={"mode": "tag"})
print(result_tag["guarded_text"])

### Mode: `i_tag` — indexed bracket labels

In [ ]:
result_itag = guard.text(sample_text, config={"mode": "i_tag"})
print(result_itag["guarded_text"])

### Inspect findings from `guard.text()`

In [ ]:
for f in result["findings"]:
    print(f"{f['type']:<20} original='{f['original_text']}'  score={f['score']:.2f}")

## 4. Work with documents

`analyze.doc()` and `guard.doc()` accept `.txt`, `.pdf`, and `.docx` file paths.
They extract text automatically and run the same pipelines as their `.text()` counterparts.

The sample files are in `examples/files/`.

In [ ]:
import os

FILES_DIR = os.path.join(os.path.dirname(os.getcwd()), "examples", "files")
# If running from examples/ folder already:
if not os.path.exists(FILES_DIR):
    FILES_DIR = os.path.join(os.getcwd(), "files")

TXT_FILE  = os.path.join(FILES_DIR, "medisch_verslag.txt")
PDF_FILE  = os.path.join(FILES_DIR, "medisch_verslag.pdf")
DOCX_FILE = os.path.join(FILES_DIR, "medisch_verslag.docx")

print("Files found:")
for path in [TXT_FILE, PDF_FILE, DOCX_FILE]:
    print(f"  {'✓' if os.path.exists(path) else '✗'}  {path}")

### Analyze a `.txt` document

In [ ]:
txt_findings = analyze.doc(TXT_FILE)

print(f"Found {len(txt_findings)} PII entities in medisch_verslag.txt:")
for f in txt_findings:
    print(f"  {f['type']:<20} score={f['score']:.2f}")

### Guard a `.pdf` document

In [ ]:
pdf_result = guard.doc(PDF_FILE, config={"mode": "tag"})

print("Guarded PDF text (first 600 chars):")
print(pdf_result["guarded_text"][:600])

### Guard a `.docx` document

In [ ]:
docx_result = guard.doc(DOCX_FILE, config={"mode": "anonymize"})

print("Guarded DOCX text (first 600 chars):")
print(docx_result["guarded_text"][:600])

## 5. Custom patterns

Use `custom_pattern()` to add your own regex-based entity types. Pass them via `config["custom_patterns"]`.

| Param | Description |
|-------|-------------|
| `name` | Entity label for this pattern |
| `regex` | Python regex |
| `score` | Confidence score (default `0.85`) |
| `context` | Words near the match that boost confidence |
| `anonymize_list` | Custom fake-value pool for `anonymize` mode |

In [ ]:
employee_pattern = custom_pattern(
    name="EMPLOYEE_ID",
    regex=r"EMP-\d{4}",
    score=0.95,
    context=["medewerker", "employee", "personeelsnummer"],
    anonymize_list=["EMP-0001", "EMP-0002", "EMP-0003"],
)

text_with_employee = "Medewerker EMP-1234 heeft toegang tot het systeem."

findings = analyze.text(
    text_with_employee,
    config={"custom_patterns": [employee_pattern]},
)

print("Detected:", findings)

In [ ]:
guarded = guard.text(
    text_with_employee,
    config={"custom_patterns": [employee_pattern], "mode": "anonymize"},
)

print("Original:", text_with_employee)
print("Guarded: ", guarded["guarded_text"])

## 6. Config options

### Filter entities — keep only specific types

In [ ]:
findings_bsn_only = analyze.text(
    sample_text,
    config={"set_entities": {"keep": ["BSN", "IBAN_CODE"]}},
)

print("Only BSN and IBAN:")
for f in findings_bsn_only:
    print(f"  {f['type']:<20} '{sample_text[f['start']:f['end']]}'") 

### Filter entities — ignore specific types

In [ ]:
findings_no_person = analyze.text(
    sample_text,
    config={"set_entities": {"ignore": ["PERSON"]}},
)

print("All entities except PERSON:")
for f in findings_no_person:
    print(f"  {f['type']:<20} '{sample_text[f['start']:f['end']]}'") 

### Score threshold — raise the bar for detections

In [ ]:
high_confidence = analyze.text(sample_text, config={"score_threshold": 0.8})
all_detections  = analyze.text(sample_text, config={"score_threshold": 0.0})

print(f"All detections (threshold=0.0): {len(all_detections)} findings")
print(f"High confidence (threshold=0.8): {len(high_confidence)} findings")

### Full medical record — end-to-end example

In [ ]:
with open(TXT_FILE, encoding="utf-8") as fh:
    medical_text = fh.read()

result = guard.text(medical_text, config={"mode": "tag"})

print("=== Original ===")
print(medical_text)
print()
print("=== Guarded (tag mode) ===")
print(result["guarded_text"])
print()
print(f"Total PII spans detected: {len(result['findings'])}")